In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
##Começo da palavra
import os

# Pasta de origem
pasta_origem = "/content/drive/MyDrive/AlvaroSampaio/sam2.2/TxtMasks"

# Lista os arquivos na pasta de origem
arquivos = os.listdir(pasta_origem)

# Itera sobre os arquivos
for arquivo in arquivos:
    # Verifica se o nome do arquivo começa com "mascara_"
    if arquivo.startswith("final_mask_"):
        novo_nome = arquivo.replace("final_mask_", "", 1)  # Remove apenas a primeira ocorrência de "mascara_"
        novo_caminho = os.path.join(pasta_origem, novo_nome)  # Define o novo caminho na mesma pasta

        # Renomeia o arquivo na mesma pasta
        os.rename(os.path.join(pasta_origem, arquivo), novo_caminho)

print("Renomeação concluída!")


Renomeação concluída!


In [ ]:
import os

def rename_files_in_folder(folder_path):
    """Renomeia arquivos removendo '_combined_mask.txt' ou '.jpg' e garantindo que termine com '.txt'."""
    for file_name in os.listdir(folder_path):
        if file_name.endswith("_complete_mask.txt") or file_name.endswith(".jpg") or file_name.endswith(".txt"):  # Verifica padrões indesejados
            base_name = file_name.replace("_complete_mask.txt", "").replace(".jpg", "").replace(".txt", "")  # Remove partes desnecessárias
            new_name = f"{base_name}.txt"  # Garante que termine com '.txt'
            old_path = os.path.join(folder_path, file_name)
            new_path = os.path.join(folder_path, new_name)

            os.rename(old_path, new_path)
            print(f"Renomeado: {file_name} -> {new_name}")

if __name__ == "__main__":
    folder_path = "/content/drive/MyDrive/AlvaroSampaio/GroudingSam2/txtmasks"  # Substitua pelo caminho da pasta
    rename_files_in_folder(folder_path)


Renomeado: evo160_jpg.rf.5b9f6748c1fedc8914585a33e84a2771.txt -> evo160_jpg.rf.5b9f6748c1fedc8914585a33e84a2771.txt
Renomeado: evo157_jpg.rf.d3a8e0aeafb140f0e0db03ae16c17558.txt -> evo157_jpg.rf.d3a8e0aeafb140f0e0db03ae16c17558.txt
Renomeado: sh6_jpg.rf.a076bfb1789d28104540ac963844b065.txt -> sh6_jpg.rf.a076bfb1789d28104540ac963844b065.txt
Renomeado: imagem68_jpg.rf.222fc4e3e66c71ec918abaad9099eec8.txt -> imagem68_jpg.rf.222fc4e3e66c71ec918abaad9099eec8.txt
Renomeado: sh6_jpg.rf.96904d75f12a1ab9ba018921afb25caa.txt -> sh6_jpg.rf.96904d75f12a1ab9ba018921afb25caa.txt
Renomeado: imagem78_jpg.rf.1f474d14590997731c09325b50771a3f.txt -> imagem78_jpg.rf.1f474d14590997731c09325b50771a3f.txt
Renomeado: imagem74_jpg.rf.8281b8b45f42ed6a42983b593b2086fc.txt -> imagem74_jpg.rf.8281b8b45f42ed6a42983b593b2086fc.txt
Renomeado: imagem73_jpg.rf.541c92df1a87536c839dca011eb2c36b.txt -> imagem73_jpg.rf.541c92df1a87536c839dca011eb2c36b.txt
Renomeado: 0-11-24_png_jpg.rf.0914ae816085b671b01c387df2dd7787.txt -

##Calculando a IOU

In [ ]:
import os
import numpy as np
import cv2  # Para redimensionamento seguro

def read_txt(file_path):
    """Lê um arquivo .txt e converte para matriz NumPy."""
    with open(file_path, 'r') as file:
        lines = file.readlines()
    return np.array([[int(pixel) for pixel in line.strip().split()] for line in lines], dtype=np.uint8)

def binarize_mask(mask):
    """Converte máscara para valores binários (0 ou 1)."""
    return (mask > 0).astype(np.uint8)

def resize_mask(mask, target_shape=(640, 640)):
    """Redimensiona a máscara para 640x640 usando vizinho mais próximo."""
    return cv2.resize(mask, (target_shape[1], target_shape[0]), interpolation=cv2.INTER_NEAREST)

def calculate_iou(seg1, seg2):
    """Calcula a IoU entre duas máscaras binárias."""
    intersection = np.logical_and(seg1, seg2)
    union = np.logical_or(seg1, seg2)

    if np.sum(union) == 0:
        return 0.0

    return np.sum(intersection) / np.sum(union)

def calculate_and_save_iou(input_folder1, input_folder2, output_folder):
    """Calcula a IoU entre máscaras correspondentes de duas pastas e salva os resultados."""

    os.makedirs(output_folder, exist_ok=True)

    for file_name in os.listdir(input_folder1):
        if not file_name.endswith('.txt'):
            continue

        input_path = os.path.join(input_folder1, file_name)
        output_path = os.path.join(input_folder2, file_name)

        if not os.path.exists(input_path) or not os.path.exists(output_path):
            print(f"⚠️ Arquivo ausente: {file_name}. Pulando...")
            continue

        seg1 = read_txt(input_path)
        seg2 = read_txt(output_path)

        # Redimensiona e binariza
        seg1 = binarize_mask(resize_mask(seg1, (640, 640)))
        seg2 = binarize_mask(resize_mask(seg2, (640, 640)))

        # Debug: imprime se a máscara estiver vazia
        if np.sum(seg1) == 0 or np.sum(seg2) == 0:
            print(f"❌ Máscara vazia em {file_name} (seg1: {np.sum(seg1)}, seg2: {np.sum(seg2)})")

        # Calcula IoU
        iou = calculate_iou(seg1, seg2)

        # Salva resultado
        iou_file_path = os.path.join(output_folder, file_name.replace('.txt', '_iou.txt'))
        with open(iou_file_path, 'w') as file:
            file.write(str(iou))

        print(f"✅ IoU calculado para {file_name}: {iou:.6f}")

if __name__ == "__main__":
    input_folder1 = "/content/drive/MyDrive/AlvaroSampaio/DetectronTeste2/Floresta/mask_rcnn_R_50_C4_1x/2025-02-18-17-18-03/TxtMasks"
    input_folder2 = "/content/drive/MyDrive/AlvaroSampaio/Iou1/TxtMasks"
    output_folder = "/content/drive/MyDrive/AlvaroSampaio/IoUModelosOriginais/mask_rcnn_R_50_C4_1x"

    calculate_and_save_iou(input_folder1, input_folder2, output_folder)


✅ IoU calculado para alina10_jpg.rf.0020cad06a387c0b3dfb42e4f24ffc81.txt: 0.867870
✅ IoU calculado para evo4248_jpg.rf.003ba3bb2bb66f0aa43edaffe198723f.txt: 0.811664
✅ IoU calculado para Cafezal-01_MP4-5_jpg.rf.00c6ab9c1cd533f5fb5bb38e106a57c4.txt: 0.911001
✅ IoU calculado para 0-57-29_png_jpg.rf.0069b1a0f7a6b4743c7b2c11ceb10678.txt: 0.422692
✅ IoU calculado para 0-44-58_png_jpg.rf.0178a55e5a86ef02db9cf561e7563e80.txt: 0.487231
✅ IoU calculado para IJ_08_25_23_2560_880_png_jpg.rf.003b540b4ff368d6fdabe775d5089fe5.txt: 0.928132
✅ IoU calculado para evo3377_jpg.rf.00c887dfefc4b0a5eb81202f47146006.txt: 0.792430
✅ IoU calculado para 2aml36_jpg.rf.01dfe2dea5c3e6da34c6755f8f0f1301.txt: 0.551006
✅ IoU calculado para alina124_jpg.rf.032ba32e04e633668c61a22b0caa2144.txt: 0.868538
✅ IoU calculado para alina_1016_jpg.rf.0370ea29a36520a45682f01de495aa31.txt: 0.833206
✅ IoU calculado para 2aml0_jpg.rf.023e9bc5e251dd890b230860ebb5432f.txt: 0.299965
✅ IoU calculado para 291214_sat_37_jpg.rf.01d3a915bf

In [ ]:
import os
import numpy as np

def calcular_media_iou(pasta_iou):
    valores_iou = []
    for nome_arquivo in os.listdir(pasta_iou):
        if nome_arquivo.endswith("_iou.txt"):
            caminho_arquivo = os.path.join(pasta_iou, nome_arquivo)
            with open(caminho_arquivo, 'r') as arquivo:
                iou = float(arquivo.read())  # Lê o valor de IoU do arquivo.
                valores_iou.append(iou)

    media_iou = np.mean(valores_iou)  # Calcula a média dos valores de IoU.

    return media_iou

# Caminho para a pasta contendo os arquivos de IoU
pasta_iou = "/content/drive/MyDrive/AlvaroSampaio/IoUModelosOriginais/mask_rcnn_R_50_C4_1x"

# Calcula e imprime a média da IoU
media_iou = calcular_media_iou(pasta_iou)
print(f"Média da IoU: {media_iou}")